# Charts: Connectivity Across South Africa

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

df = pd.read_csv(Path('..') / 'data' / '20_processed' / 'census2022_household_analysis_226.csv',
                 dtype={'QID': str})
# CSV dropped the ordered category from pd.cut: restore it so tables/charts sort correctly
df['head_age_band'] = pd.Categorical(
    df['head_age_band'], categories=['<18', '18-34', '35-49', '50-64', '65+'], ordered=True)

# ── Brand plotly template ─────────────────────────────────────────────
# Colours mirror the brand tokens in styles.css; white background, no gridlines.
PALETTE = ['#65A6BE', '#75B8BE', '#C997AF', '#B9B0D3', '#F4CF97', '#97B9A0', '#F6DDCD']
brand = go.layout.Template()
brand.layout = go.Layout(
    font=dict(family='Roboto, sans-serif', size=13, color='#111111'),
    title_font=dict(size=16, color='#104C52'),   # --color-teal-dark
    colorway=PALETTE,
    paper_bgcolor='white',
    plot_bgcolor='white',
    xaxis=dict(showgrid=False, zeroline=False),
    yaxis=dict(showgrid=False, zeroline=False),
)
pio.templates['brand'] = brand
pio.templates.default = 'brand'

## Task 4 — `go.Bar`: ownership rate by province

Cellphone ownership rate (share answering “Yes”) by province.

In [2]:
#| label: fig-cellphone
#| fig-cap: "Households with a cellphone, by province (%)"

rates = (df.assign(has_cell=df['H12_CELLPHONE'].eq('Yes'))
           .groupby('province_name')['has_cell'].mean().mul(100).round(1).reset_index())
fig = go.Figure(go.Bar(
    x=rates['province_name'],
    y=rates['has_cell'],
))
fig.update_layout(title='Cellphone ownership by province', yaxis_title='%')
fig.show()

## Task 5 — Tabset: the same indicator three ways

Internet access rate by province, area type, and zone, shown as tabs. The helper keeps the three cells identical bar the grouping column.

In [3]:
def internet_rate(col):
    return (df.dropna(subset=['has_internet'])
              .groupby(col)['has_internet'].mean().mul(100).round(1).reset_index())

::: {.panel-tabset}

## By province

In [4]:
r = internet_rate('province_name')
fig = go.Figure(go.Bar(x=r['province_name'], y=r['has_internet']))
fig.update_layout(title='Internet access by province', yaxis_title='%')
fig.show()

## By area type

In [5]:
# Tab: By area type
r = internet_rate('Geo_type')
fig = go.Figure(go.Bar(x=r['Geo_type'], y=r['has_internet']))
fig.update_layout(title='Internet access by area type', yaxis_title='%')
fig.show()

## By zone

In [6]:
# Tab: By zone
r = internet_rate('zone')
fig = go.Figure(go.Bar(x=r['zone'], y=r['has_internet']))
fig.update_layout(title='Internet access by zone', yaxis_title='%')
fig.show()

:::

## Task 6: Cross-references

As shown in @fig-cellphone, cellphone ownership is high across all provinces. The connectivity table on the Tables page gives the full provincial breakdown.

## Task 7 — `go.Histogram`

Distribution of household-head age.

In [7]:
fig = go.Figure(go.Histogram(x=df['DERH_HHAGE'], nbinsx=25))
fig.update_layout(title='Age of household head', xaxis_title='Age', yaxis_title='Households')
fig.show()

## Task 8 — Stacked bar: the full internet-access mix

A 100% stacked bar shows *composition*, not just a rate: every household's `H13_INTERNET_ACCESS` mode, by province. This is the full breakdown that sits behind the binary `has_internet` flag.

In [8]:
#| label: fig-internet-mix
#| fig-cap: "Internet-access mode composition by province (%)"

mix = (pd.crosstab(df['province_name'], df['H13_INTERNET_ACCESS'], normalize='index') * 100).round(1)
fig = go.Figure()
for mode in mix.columns:
    fig.add_trace(go.Bar(x=mix.index, y=mix[mode], name=mode))
fig.update_layout(
    barmode='stack',
    title='How households access the internet, by province',
    yaxis_title='% of households', legend_title='Access mode',
)
fig.show()

## Task 9 — Bubble chart: two rates at once

A `go.Scatter` in `markers` mode becomes a **bubble chart** when the marker *size* encodes a third variable. Here each province is one point — cellphone vs internet access — sized by the number of households.

In [9]:
#| label: fig-connectivity-bubble
#| fig-cap: "Cellphone vs internet access by province (bubble area = households)"

prov = (df.assign(has_cell=df['H12_CELLPHONE'].eq('Yes'))
          .groupby('province_name')
          .agg(cell_pct=('has_cell', 'mean'),
               net_pct=('has_internet', 'mean'),
               households=('QID', 'size'))
          .reset_index())
prov[['cell_pct', 'net_pct']] = prov[['cell_pct', 'net_pct']] * 100

fig = go.Figure(go.Scatter(
    x=prov['cell_pct'], y=prov['net_pct'],
    mode='markers+text', text=prov['province_name'], textposition='top center',
    marker=dict(
        size=prov['households'],
        sizemode='area',
        sizeref=2. * prov['households'].max() / (45. ** 2),
        sizemin=4, color='#65A6BE'),
))
fig.update_layout(title='Connectivity by province',
                  xaxis_title='Cellphone ownership (%)', yaxis_title='Internet access (%)')
fig.show()

## Reflection

- What did `go.Bar` make you specify by hand that `plotly.express` would have inferred automatically?
- When is the full `H13_INTERNET_ACCESS` breakdown (the stacked bar) more informative than the binary `has_internet` flag?
- In the bubble chart, what does encoding household count as marker *area* — rather than as a fourth bar chart — let the reader see at a glance?